In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits


In [ ]:
# Find the project root whether Jupyter starts here, in the notebook folder, or its parent.
cwd = Path.cwd().resolve()
candidates = [cwd, *cwd.parents]
candidates += [folder / "applied-ml-midterm" for folder in candidates.copy()]
PROJECT_DIR = next((folder for folder in candidates if (folder / "part1-data").is_dir()), None)
if PROJECT_DIR is None:
    raise FileNotFoundError(f"Could not find part1-data from {cwd}")
DATA_DIR = PROJECT_DIR / "part1-data"
OUTPUT_DIR = PROJECT_DIR / "deliverables" / "reproduced" / "part1_supplementary"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

part1_train = pd.read_csv(DATA_DIR / "part1_train.csv")
part1_val = pd.read_csv(DATA_DIR / "part1_val.csv")
part1_test = pd.read_csv(DATA_DIR / "part1_test.csv")
REASONS = ["Safety", "Funding", "Efficacy", "Recruitment", "Other"]
EMBEDDINGS = [f"emb_{i}" for i in range(150)]
print("Train, validation, test:", len(part1_train), len(part1_val), len(part1_test))


In [ ]:
def scale_embeddings(train_df, val_df, test_df):
    """Fit the scaler on training embeddings only."""
    scaler = StandardScaler()
    X_train = scaler.fit_transform(train_df[EMBEDDINGS])
    X_val = scaler.transform(val_df[EMBEDDINGS])
    X_test = scaler.transform(test_df[EMBEDDINGS])
    return X_train, X_val, X_test, scaler


def observed_label_mask(train_df):
    known = train_df[REASONS].isin([0, 1]).all(axis=1).to_numpy()
    hidden = train_df[REASONS].eq(-100).all(axis=1).to_numpy()
    if not (known | hidden).all():
        raise ValueError("Expected each training trial to have five observed or five masked labels.")
    return known, hidden


def cluster_reason_scores(X_train, train_df, known, n_clusters=10,
                          smoothing_strength=2.0, random_state=42):
    """K-means on every training embedding; rates use observed labels only."""
    y_known = train_df.loc[known, REASONS].to_numpy(dtype=int)
    model = KMeans(n_clusters=n_clusters, init="k-means++", n_init=10,
                   max_iter=300, tol=1e-4, random_state=random_state)
    with threadpool_limits(limits=1):
        cluster_ids = model.fit_predict(X_train)
    member_counts = np.bincount(cluster_ids[known], minlength=n_clusters)
    positive_counts = np.zeros((n_clusters, len(REASONS)), dtype=float)
    np.add.at(positive_counts, cluster_ids[known], y_known)
    prior = y_known.mean(axis=0)
    rates = (positive_counts + smoothing_strength * prior) / (member_counts[:, None] + smoothing_strength)
    return model, cluster_ids, rates, member_counts


def neighbor_reason_scores(X_train, train_df, known, hidden, n_neighbors=3):
    """Vote among originally observed trials in the same scaled150 space."""
    y_known = train_df.loc[known, REASONS].to_numpy(dtype=int)
    model = KNeighborsClassifier(n_neighbors=n_neighbors, weights="uniform",
                                 metric="euclidean", algorithm="brute", n_jobs=1)
    model.fit(X_train[known], y_known)
    with threadpool_limits(limits=1):
        probabilities = model.predict_proba(X_train[hidden])
    votes = np.column_stack([
        probability[:, np.flatnonzero(classes == 1)[0]]
        for probability, classes in zip(probabilities, model.classes_)
    ])
    return model, votes


def threshold_reasons(scores, threshold):
    """Apply independent multilabel threshold, filling empty rows by argmax."""
    prediction = (np.asarray(scores) >= threshold).astype(int)
    empty = prediction.sum(axis=1) == 0
    prediction[np.flatnonzero(empty), np.asarray(scores)[empty].argmax(axis=1)] = 1
    return prediction


def recover_training_labels(train_df, known, hidden, cluster_ids, cluster_rates,
                            neighbor_votes, threshold=0.30, cluster_weight=0.5):
    """Blend cluster and neighbor scores only for the 1014 masked trials."""
    cluster_scores = cluster_rates[cluster_ids[hidden]]
    scores = cluster_weight * cluster_scores + (1 - cluster_weight) * neighbor_votes
    recovered = threshold_reasons(scores, threshold)
    labels = np.empty((len(train_df), len(REASONS)), dtype=int)
    labels[known] = train_df.loc[known, REASONS].to_numpy(dtype=int)
    labels[hidden] = recovered
    assert np.isin(labels, [0, 1]).all()
    assert np.array_equal(labels[known], train_df.loc[known, REASONS].to_numpy(dtype=int))
    return labels, scores


def fit_reason_models(X_train, labels, known, C=0.1, pseudo_weight=0.25):
    """Five binary logistic regressions; observed rows have weight 1."""
    weights = np.where(known, 1.0, pseudo_weight)
    return [
        LogisticRegression(C=C, max_iter=1000, solver="liblinear", random_state=42)
        .fit(X_train, labels[:, j], sample_weight=weights)
        for j in range(len(REASONS))
    ]


def predict_reasons(models, X):
    probabilities = np.column_stack([model.predict_proba(X)[:, 1] for model in models])
    return threshold_reasons(probabilities, 0.5)


def reason_metrics(truth, prediction):
    return pd.Series({
        "Sample-wise F1": f1_score(truth, prediction, average="samples", zero_division=0),
        "Macro F1": f1_score(truth, prediction, average="macro", zero_division=0),
        **dict(zip(REASONS, f1_score(truth, prediction, average=None, zero_division=0))),
    })


In [ ]:
# Fixed teammate-selected settings; this notebook does not rerun their search grid.
X_train, X_val, X_test, scaler = scale_embeddings(part1_train, part1_val, part1_test)
known, hidden = observed_label_mask(part1_train)
assert known.sum() == 157 and hidden.sum() == 1014
kmeans_model, cluster_ids, cluster_rates, cluster_known_counts = cluster_reason_scores(
    X_train, part1_train, known, n_clusters=10, smoothing_strength=2.0, random_state=42
)
neighbor_model, neighbor_votes = neighbor_reason_scores(
    X_train, part1_train, known, hidden, n_neighbors=3
)
augmented_labels, recovery_scores = recover_training_labels(
    part1_train, known, hidden, cluster_ids, cluster_rates, neighbor_votes,
    threshold=0.30, cluster_weight=0.5
)
reason_models = fit_reason_models(
    X_train, augmented_labels, known, C=0.1, pseudo_weight=0.25
)
validation_predictions = predict_reasons(reason_models, X_val)
validation_metrics = reason_metrics(
    part1_val[REASONS].to_numpy(dtype=int), validation_predictions
)
test_predictions = predict_reasons(reason_models, X_test)
print("Validation metrics:")
print(validation_metrics.to_string())
print("Recovered positives:", dict(zip(REASONS, augmented_labels[hidden].sum(axis=0))))
print("Unresolved training trials:", 0)


In [ ]:
# Export only after checking the fixed method against the saved teammate artifacts.
submission_path = OUTPUT_DIR / "part1_teammate_hybrid_rebuilt_submission.csv"
submission = part1_test[["ID"]].copy()
submission["Predicted"] = [
    " ".join(reason for reason, value in zip(REASONS, row) if value)
    for row in test_predictions
]
assert len(submission) == len(part1_test) == 312
assert submission["ID"].equals(part1_test["ID"])
assert submission["Predicted"].notna().all()
assert submission["Predicted"].str.strip().ne("").all()
submission.to_csv(submission_path, index=False)
print("Saved:", submission_path)
submission.head()


In [ ]:
from sklearn.decomposition import KernelPCA


def kernel_pca_for_clustering(X_train, n_components=50, gamma=1/75, random_state=42):
    """Fit RBF kernel PCA on train embeddings; only K-means uses this output."""
    transformer = KernelPCA(
        n_components=n_components, kernel="rbf", gamma=gamma,
        eigen_solver="arpack", random_state=random_state,
    )
    X_cluster = transformer.fit_transform(X_train)
    return X_cluster, transformer


In [ ]:
# Controlled change: KernelPCA50 replaces raw scaled150 for KMeans only.
# Neighbor votes, recovered-label rule, and final classifier remain the baseline's.
X_train_cluster_kpca, kernel_pca_model = kernel_pca_for_clustering(
    X_train, n_components=50, gamma=1/75, random_state=42
)
kpca_kmeans_model, kpca_cluster_ids, kpca_cluster_rates, kpca_known_counts = cluster_reason_scores(
    X_train_cluster_kpca, part1_train, known,
    n_clusters=10, smoothing_strength=2.0, random_state=42,
)
kpca_augmented_labels, kpca_recovery_scores = recover_training_labels(
    part1_train, known, hidden, kpca_cluster_ids, kpca_cluster_rates,
    neighbor_votes, threshold=0.30, cluster_weight=0.5,
)
kpca_reason_models = fit_reason_models(
    X_train, kpca_augmented_labels, known, C=0.1, pseudo_weight=0.25
)
kpca_validation_predictions = predict_reasons(kpca_reason_models, X_val)
kpca_validation_metrics = reason_metrics(
    part1_val[REASONS].to_numpy(dtype=int), kpca_validation_predictions
)
kpca_test_predictions = predict_reasons(kpca_reason_models, X_test)
assert np.array_equal(kpca_augmented_labels[known], part1_train.loc[known, REASONS].to_numpy(dtype=int))
assert kpca_augmented_labels[hidden].any(axis=1).all()
print("Baseline vs KernelPCA-only clustering change:")
print(pd.DataFrame({"Baseline raw clusters": validation_metrics, "KernelPCA clusters": kpca_validation_metrics}).to_string())
print("Recovered positive counts (baseline / KernelPCA):")
print(pd.DataFrame({
    "Baseline": augmented_labels[hidden].sum(axis=0),
    "KernelPCA": kpca_augmented_labels[hidden].sum(axis=0),
}, index=REASONS).to_string())
print("Unresolved training trials:", 0)
print("Raw empty validation/test before final fallback:",
      int((np.column_stack([m.predict_proba(X_val)[:, 1] for m in kpca_reason_models]) < 0.5).all(axis=1).sum()),
      int((np.column_stack([m.predict_proba(X_test)[:, 1] for m in kpca_reason_models]) < 0.5).all(axis=1).sum()))


In [ ]:
# Separate Kaggle-formatted output for the KernelPCA clustering experiment.
kpca_submission_path = OUTPUT_DIR / "part1_teammate_hybrid_kernelpca_clusters_submission.csv"
kpca_submission = part1_test[["ID"]].copy()
kpca_submission["Predicted"] = [
    " ".join(reason for reason, value in zip(REASONS, row) if value)
    for row in kpca_test_predictions
]
assert len(kpca_submission) == len(part1_test) == 312
assert kpca_submission["ID"].equals(part1_test["ID"])
assert kpca_submission["Predicted"].notna().all()
assert kpca_submission["Predicted"].str.strip().ne("").all()
kpca_submission.to_csv(kpca_submission_path, index=False)
print("Saved:", kpca_submission_path)
kpca_submission.head()


In [ ]:
def run_kernel_pca_dimension(n_components):
    """Change only clustering representation; reuse baseline NN votes and LR inputs."""
    X_cluster, transformer = kernel_pca_for_clustering(
        X_train, n_components=n_components, gamma=1/75, random_state=42
    )
    cluster_model, cluster_ids, rates, labeled_counts = cluster_reason_scores(
        X_cluster, part1_train, known,
        n_clusters=10, smoothing_strength=2.0, random_state=42,
    )
    labels, scores = recover_training_labels(
        part1_train, known, hidden, cluster_ids, rates, neighbor_votes,
        threshold=0.30, cluster_weight=0.5,
    )
    models = fit_reason_models(X_train, labels, known, C=0.1, pseudo_weight=0.25)
    predictions = predict_reasons(models, X_val)
    metrics = reason_metrics(part1_val[REASONS].to_numpy(dtype=int), predictions)
    return {
        "n_components": n_components,
        "transformer": transformer,
        "cluster_model": cluster_model,
        "labels": labels,
        "validation_predictions": predictions,
        "validation_metrics": metrics,
        "recovered_positive_counts": labels[hidden].sum(axis=0),
        "unresolved_rows": 0,
    }


In [ ]:
# Prespecified KernelPCA dimensions; raw150 teammate baseline is a separate reference row.
KPCA_DIMENSIONS = [20, 30, 50, 75, 100, 150]
kpca_dimension_results = {}
kpca_sweep_rows = [{
    "Clustering representation": "Raw scaled150 baseline",
    **validation_metrics.to_dict(),
    **{f"Recovered {r}": int(augmented_labels[hidden, j].sum()) for j, r in enumerate(REASONS)},
    "Unresolved rows": 0,
    "Changed recovered rows vs baseline": 0,
}]
for n_components in KPCA_DIMENSIONS:
    run = run_kernel_pca_dimension(n_components)
    kpca_dimension_results[n_components] = run
    kpca_sweep_rows.append({
        "Clustering representation": f"RBF KernelPCA {n_components}",
        **run["validation_metrics"].to_dict(),
        **{f"Recovered {r}": int(run["recovered_positive_counts"][j]) for j, r in enumerate(REASONS)},
        "Unresolved rows": run["unresolved_rows"],
        "Changed recovered rows vs baseline": int(np.any(run["labels"] != augmented_labels, axis=1).sum()),
    })

# The independently fitted 50-dimensional experiment must be reproduced exactly.
assert np.array_equal(kpca_dimension_results[50]["labels"], kpca_augmented_labels)
assert np.array_equal(kpca_dimension_results[50]["validation_predictions"], kpca_validation_predictions)
kpca_dimension_table = pd.DataFrame(kpca_sweep_rows).set_index("Clustering representation")
print("Prespecified clustering-representation sweep (no model selected):")
print(kpca_dimension_table.to_string(float_format=lambda x: f"{x:.4f}"))


In [ ]:
from sklearn.metrics.pairwise import rbf_kernel


def kernel_kmeans_train(X, n_clusters=10, gamma=1/75, n_init=10,
                        max_iter=100, random_state=42):
    """Train kernel K-means in the RBF feature space and return train assignments.

    d(i,C)=K(i,i)-2 mean_j K(i,j)+mean_{j,l} K(j,l). No validation/test
    point is used; this experiment needs train assignments for label recovery only.
    """
    X = np.asarray(X)
    if len(X) < n_clusters:
        raise ValueError("Number of clusters exceeds training rows.")
    with threadpool_limits(limits=1):
        gram = rbf_kernel(X, X, gamma=gamma)
    diagonal = np.diag(gram)
    row_indices = np.arange(len(X))
    rng = np.random.default_rng(random_state)

    def distances(labels):
        membership = np.eye(n_clusters)[labels]
        sizes = membership.sum(axis=0)
        if (sizes == 0).any():
            raise ValueError("Empty cluster before distance update.")
        kernel_sums = gram @ membership
        centroid_norms = (membership * kernel_sums).sum(axis=0) / sizes**2
        d = diagonal[:, None] - 2 * kernel_sums / sizes + centroid_norms
        return np.maximum(d, 0.0)

    def repair_empty(labels, old_distances):
        counts = np.bincount(labels, minlength=n_clusters)
        for empty in np.flatnonzero(counts == 0):
            eligible = counts[labels] > 1
            assigned_distance = old_distances[row_indices, labels].copy()
            assigned_distance[~eligible] = -np.inf
            donor = int(np.argmax(assigned_distance))
            if not np.isfinite(assigned_distance[donor]):
                raise RuntimeError("Could not repair empty kernel K-means cluster.")
            counts[labels[donor]] -= 1
            labels[donor] = empty
            counts[empty] += 1
        return labels

    def initialize():
        # Kernel K-means++: pick data points using squared RKHS distance.
        chosen = [int(rng.integers(len(X)))]
        nearest = diagonal + diagonal[chosen[0]] - 2 * gram[:, chosen[0]]
        for _ in range(1, n_clusters):
            nearest = np.maximum(nearest, 0.0)
            nearest[chosen] = 0.0
            if nearest.sum() == 0:
                candidates = np.setdiff1d(row_indices, chosen)
                candidate = int(rng.choice(candidates))
            else:
                candidate = int(rng.choice(len(X), p=nearest / nearest.sum()))
            chosen.append(candidate)
            candidate_dist = diagonal + diagonal[candidate] - 2 * gram[:, candidate]
            nearest = np.minimum(nearest, candidate_dist)
        seed_distances = np.column_stack([
            diagonal + diagonal[j] - 2 * gram[:, j] for j in chosen
        ])
        labels = seed_distances.argmin(axis=1)
        return repair_empty(labels, seed_distances)

    best = None
    start_summaries = []
    for start in range(n_init):
        labels = initialize()
        converged = False
        for iteration in range(1, max_iter + 1):
            d = distances(labels)
            updated = repair_empty(d.argmin(axis=1), d)
            if np.array_equal(updated, labels):
                converged = True
                break
            labels = updated
        final_distances = distances(labels)
        objective = float(final_distances[row_indices, labels].sum())
        counts = np.bincount(labels, minlength=n_clusters)
        assert (counts > 0).all()
        start_summaries.append({"start": start, "objective": objective,
                                "iterations": iteration, "converged": converged})
        if best is None or objective < best["objective"]:
            best = {"labels": labels.copy(), "objective": objective,
                    "iterations": iteration, "converged": converged,
                    "cluster_sizes": counts.copy(), "start": start}
    best["starts"] = pd.DataFrame(start_summaries)
    return best


def smoothed_rates_for_assignments(cluster_ids, train_df, known,
                                   n_clusters=10, smoothing_strength=2.0):
    """Same observed-only cluster-rate formula as the teammate baseline."""
    y_known = train_df.loc[known, REASONS].to_numpy(dtype=int)
    member_counts = np.bincount(cluster_ids[known], minlength=n_clusters)
    positive_counts = np.zeros((n_clusters, len(REASONS)), dtype=float)
    np.add.at(positive_counts, cluster_ids[known], y_known)
    prior = y_known.mean(axis=0)
    rates = (positive_counts + smoothing_strength * prior) / (member_counts[:, None] + smoothing_strength)
    return rates, member_counts


In [ ]:
# Actual kernel K-means, not KernelPCA followed by Euclidean K-means.
kernel_kmeans_result = kernel_kmeans_train(
    X_train, n_clusters=10, gamma=1/75, n_init=10,
    max_iter=100, random_state=42,
)
kernel_cluster_ids = kernel_kmeans_result["labels"]
kernel_cluster_rates, kernel_known_counts = smoothed_rates_for_assignments(
    kernel_cluster_ids, part1_train, known,
    n_clusters=10, smoothing_strength=2.0,
)
kernel_augmented_labels, kernel_recovery_scores = recover_training_labels(
    part1_train, known, hidden, kernel_cluster_ids, kernel_cluster_rates,
    neighbor_votes, threshold=0.30, cluster_weight=0.5,
)
kernel_reason_models = fit_reason_models(
    X_train, kernel_augmented_labels, known, C=0.1, pseudo_weight=0.25,
)
kernel_validation_predictions = predict_reasons(kernel_reason_models, X_val)
kernel_validation_metrics = reason_metrics(
    part1_val[REASONS].to_numpy(dtype=int), kernel_validation_predictions,
)
kernel_test_predictions = predict_reasons(kernel_reason_models, X_test)
assert np.array_equal(kernel_augmented_labels[known], part1_train.loc[known, REASONS].to_numpy(dtype=int))
assert kernel_augmented_labels[hidden].any(axis=1).all()
print("Teammate raw KMeans / KernelPCA50+KMeans / actual kernel KMeans:")
print(pd.DataFrame({
    "Raw KMeans": validation_metrics,
    "KernelPCA50 + KMeans": kpca_validation_metrics,
    "Kernel KMeans": kernel_validation_metrics,
}).to_string())
print("Recovered positives:", dict(zip(REASONS, kernel_augmented_labels[hidden].sum(axis=0))))
print("Changed recovered rows vs raw KMeans:", int(np.any(kernel_augmented_labels != augmented_labels, axis=1).sum()))
print("Unresolved training trials:", 0)
print("Kernel KMeans best start/objective/iterations/converged:",
      kernel_kmeans_result["start"], kernel_kmeans_result["objective"],
      kernel_kmeans_result["iterations"], kernel_kmeans_result["converged"])
print("Cluster sizes:", kernel_kmeans_result["cluster_sizes"].tolist())
print("Original labeled counts:", kernel_known_counts.tolist())
print("Start diagnostics:")
print(kernel_kmeans_result["starts"].to_string(index=False))


In [ ]:
# Separate CSV for comparing actual kernel K-means test assignments.
kernel_submission_path = OUTPUT_DIR / "part1_teammate_hybrid_kernel_kmeans_submission.csv"
kernel_submission = part1_test[["ID"]].copy()
kernel_submission["Predicted"] = [
    " ".join(reason for reason, value in zip(REASONS, row) if value)
    for row in kernel_test_predictions
]
assert len(kernel_submission) == len(part1_test) == 312
assert kernel_submission["ID"].equals(part1_test["ID"])
assert kernel_submission["Predicted"].notna().all()
assert kernel_submission["Predicted"].str.strip().ne("").all()
kernel_submission.to_csv(kernel_submission_path, index=False)
print("Saved:", kernel_submission_path)
kernel_submission.head()
